In [5]:
import pandas as pd
SEED=42
# Prefer a local data/ folder for portability.
DATA_PATH = "data/7817_1.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

df.sample(5, random_state=SEED)

Dataset shape: (1597, 27)


,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
802,AVpe7LD5LJeJML43ybWA,"B00DOPNO4M,B00BWYQ9YE,B00CYQPMJC,B00CUU1CGY,B0...",Amazon,"Amazon Devices,Kindle Store,buy a kindle",NaN,2015-05-22T15:33:59Z,2017-07-18T23:52:40Z,NaN,NaN,"kindlefirehdx7/b00dopno4m,kindlefirehdx7/b00bw...",...,NaN,http://www.amazon.com/kindle-fire-hdx-student-...,I love this handheld device especially all the...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,AVpfpzCi1cnluZ0-oxEr,B00DOPNLJ0,Amazon,Amazon Devices,NaN,2015-06-02T08:44:19Z,2017-08-07T15:41:59Z,NaN,NaN,kindlefirehdx89/b00dopnlj0,...,NaN,http://www.amazon.com/Kindle-Fire-HDX-Display-...,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,NaN,NaN,B. Tarbuck,NaN,NaN,NaN
350,AVpfLiCSilAPnD_xWpk_,B00CX5P8FC,Amazon,"Categories,Amazon Devices,Electronics Features...",NaN,2015-05-22T18:12:20Z,2017-08-08T22:03:26Z,NaN,8.487190e+11,"848719022827,0848719022827,amazonfiretv/b00cx5...",...,NaN,http://www.amazon.com/Fire-TV-streaming-media-...,This was easy to set up I can access many movi...,Amazon Fire TV - A must have!!!!,NaN,NaN,Amazon Customer,NaN,8.487190e+11,NaN
682,AVzRlo37glJLPUi8FbPy,B01LW1MS9C,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:46Z,NaN,NaN,amazonechodotcasefitsechodot2ndgenerationonlyc...,...,5.0,https://www.amazon.com/Amazon-Echo-Case-fits-G...,I am thoroughly impressed with my Echo Dot and...,The Extra Touch,NaN,NaN,dm,NaN,NaN,NaN
1324,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,5.0,http://reviews.bestbuy.com/3545/5097300/review...,Great little device easy to connect to bluetoo...,Easy small decent sound,NaN,NaN,Drjim,NaN,8.416670e+11,1.75 lbs


### Load Models (Your saved model)

In [1]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, DistilBertModel
import torch

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("sentiment_model")

# Load your trained sentiment model
sentiment_model = DistilBertForSequenceClassification.from_pretrained("sentiment_model")

# Load base model for embeddings
embedding_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

sentiment_model.eval()
embedding_model.eval()

d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1351.29it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

### Embedding Function

In [7]:
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = embedding_model(**inputs)

    # CLS token embedding (important)
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding.squeeze().numpy()

### Create Embeddings for Dataset

In [9]:
texts = df['reviews.text'].tolist()

embeddings = [get_embedding(text) for text in texts]

### Apply KMeans Clustering

In [10]:
from sklearn.cluster import KMeans

num_clusters = 4  # you can tune this
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

clusters = kmeans.fit_predict(embeddings)

### Map Clusters → Aspects

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
X_tfidf = tfidf.fit_transform(texts)

feature_names = np.array(tfidf.get_feature_names_out())

def get_cluster_keywords(texts, clusters, X_tfidf, feature_names, top_n=3):
    cluster_keywords = {}

    for cluster_id in np.unique(clusters):
        # Get indices of texts in this cluster
        idx = np.where(clusters == cluster_id)[0]
        
        # Average TF-IDF scores for cluster
        cluster_tfidf = X_tfidf[idx].mean(axis=0)
        
        # Get top words
        top_indices = np.argsort(cluster_tfidf).A1[::-1][:top_n]
        keywords = feature_names[top_indices]
        
        cluster_keywords[cluster_id] = ", ".join(keywords)

    return cluster_keywords

In [15]:
cluster_map = get_cluster_keywords(texts, clusters, X_tfidf, feature_names)

print(cluster_map)

{np.int32(0): 'kindle, amazon, prime', np.int32(1): 'great, tap, echo', np.int32(2): 'headphones, kindle, like', np.int32(3): 'great, love, easy'}


### Sentiment Prediction Function

In [16]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = sentiment_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label_map = {0: "Negative 😡", 1: "Positive 😊"}

    return label_map[pred]

### Final ASBA Function

In [17]:
def aspect_based_analysis(text):
    # Get embedding
    emb = get_embedding(text)

    # Predict cluster
    cluster = kmeans.predict([emb])[0]
    aspect = cluster_map.get(cluster, "Unknown")

    # Predict sentiment
    sentiment = predict_sentiment(text)

    return {
        "Review": text,
        "Aspect": aspect,
        "Sentiment": sentiment
    }

In [18]:
print(aspect_based_analysis("Delivery was late but the product quality is amazing"))
print(aspect_based_analysis("Very expensive product but packaging was good"))

{'Review': 'Delivery was late but the product quality is amazing', 'Aspect': 'great, love, easy', 'Sentiment': 'Negative 😡'}
{'Review': 'Very expensive product but packaging was good', 'Aspect': 'great, love, easy', 'Sentiment': 'Negative 😡'}
